# tuple_methods
Tuples are small, fixed-shape sequence containers. They shine when structure should be stable: coordinates, compound keys, parser results, and multi-value returns. The engineering value is semantic clarity around immutability and shape, not just syntax.

## 1. Problem First
Why does this matter in real systems?
- Functions often return multiple values as tuples.
- Compound keys in dictionaries are often tuples.
- Stable shapes reduce accidental mutation of record structure.

In [ ]:
endpoint = ("api", "us-east-1", 443)
service, region, port = endpoint
print(service, region, port)

## 2. Minimal Concept
Tuple tools and patterns:
- Methods: `count()`, `index()`
- Packing and unpacking
- Nested tuples and slicing
- Concatenation, repetition, membership
- Conversion list ↔ tuple
- Built-ins: `max()`, `min()`, `sum()`, `len()`, `enumerate()`, `zip()`
- Use cases: swapping, multi-return, dict keys, `namedtuple`

## 3. Mental Model
How Python actually behaves
A tuple is immutable in structure, so you cannot append, assign, or remove elements. But immutability is shallow: if a tuple contains a list or dict, that inner object can still mutate. Tuples are also hashable only if all their elements are hashable.

In [ ]:
payload = ("job", [1, 2])
payload[1].append(3)
print(payload)
print(payload.count("job"), payload.index("job"))

## 4. Internal Mechanics
Tuples expose very few methods because they are not meant for structural mutation. Their power comes from stable position-based access, unpacking, and safe use as keys when contents are hashable.

In [ ]:
from collections import namedtuple
import dis

def build_pair(a, b):
    return (a, b)

dis.dis(build_pair)
Point = namedtuple("Point", ["x", "y"])
p = Point(10, 20)
print(p.x, p.y)
cache = {("api", "us"): "ok"}
print(cache[("api", "us")])

## 5. Edge Cases
Where it breaks:
- `(1)` is an `int`, not a single-element tuple.
- Tuples containing mutable values are not fully “safe.”
- A tuple containing an unhashable element cannot be used as a dict key.
- Long positional tuples can become unclear compared to named structures.

In [ ]:
single = (1)
real_single = (1,)
print(type(single), type(real_single))
nested = ((1, 2), (3, 4))
print(nested[0:1])
print((1, 2) + (3, 4))
print(("x",) * 3)
print(2 in (1, 2, 3))
try:
    bad = {([1, 2],): "x"}
except TypeError as exc:
    print(type(exc).__name__)

## 6. Performance Thinking
- Tuple indexing is O(1).
- Tuples are often slightly lighter than lists because they do not resize.
- Hashable tuples enable O(1) average dictionary lookup for compound keys.
- Their biggest benefit is usually semantic correctness, not dramatic speed.

## 7. Real Use Case
Tuples are ideal for multi-value returns, swapping variables, and using compound identifiers as dictionary keys.

In [ ]:
def parse_status(line):
    service, status = line.split(":")
    return service, status

service, status = parse_status("api:ok")
print(service, status)
a, b = 10, 20
a, b = b, a
print(a, b)
values = tuple([3, 1, 4])
print(max(values), min(values), sum(values), len(values))
print(list(enumerate(values)))
print(list(zip(values, ("a", "b", "c"))))

## 8. Anti-Pattern
What beginners do wrong:
- Use long tuples where field names matter.
- Assume tuples prevent all nested mutation.
- Forget the trailing comma for single-item tuples.
- Store mutable internals and still call the whole structure “immutable.”

In [ ]:
response = (200, "ok", {"latency": 120})
response[2]["latency"] = 500
print(response)
print(tuple([1, 2, 3]))
print(list((1, 2, 3)))

## 9. Interview Signals
Questions FAANG asks:
- Why can tuples be used as dict keys while lists cannot?
- Is a tuple always fully immutable?
- When should you use `namedtuple` instead of a raw tuple?
- What problem does tuple unpacking solve cleanly?

## 10. Exercise (Non-trivial)
Design a caching layer keyed by `(service, region, version)`. Return multiple values from a parser as tuples, use unpacking at call sites, and decide where raw tuples stop being expressive enough and `namedtuple` or a dataclass becomes the better choice.

In [ ]:
def build_cache_key(service, region, version):
    return service, region, version

# Task:
# 1. Show how this key is used in a dictionary.
# 2. Explain hashability constraints.
# 3. Compare a raw tuple key to namedtuple for readability.